# Pipeline d'Extraction des Données Strava

Ce notebook détaille les différentes étapes de l'extraction des données depuis l'API Strava jusqu'à leur sauvegarde sur le stockage cloud Onyxia (MinIO).

## Étape 1 : Initialisation et Connexion

Dans cette première étape, nous chargeons nos clés d'authentification depuis le fichier `.env` et nous définissons les chemins vers nos données.

## Étape 2 : L'Authentification Strava

Pour interagir avec l'API Strava, nous avons besoin d'un "Access Token" (jeton d'accès) temporaire. Nous l'obtenons en échangeant notre "Refresh Token" (jeton de rafraîchissement) qui a une durée de vie plus longue.

**Important :** Si tu n'as pas de Refresh Token valide, tu devras d'abord exécuter le script `auth.py` pour autoriser l'application.

In [6]:
import os
import time
import requests
import pandas as pd
import s3fs
from pathlib import Path
from dotenv import load_dotenv

# 1. Définition du chemin racine (depuis le dossier notebooks)
# Si tu lances ce notebook depuis /notebooks, la racine est le dossier parent.
ROOT_DIR = Path().resolve().parent

# 2. Chargement du fichier .env
load_dotenv(dotenv_path=ROOT_DIR / ".env")




True

## Étape 3 : L'Extraction des Activités

Maintenant que nous sommes authentifiés, nous pouvons interroger l'API pour récupérer l'historique complet des activités. L'API renvoie les données par "pages" (pagination). Nous devons donc faire plusieurs requêtes pour tout récupérer.

In [8]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Remonte à la racine du projet
# ROOT_DIR est déjà défini dans la cellule précédente
load_dotenv(dotenv_path=ROOT_DIR / ".env")

def get_new_access_token():
    """Échange le refresh_token contre un access_token tout neuf"""
    
    payload = {
        'client_id': os.getenv('STRAVA_CLIENT_ID'),
        'client_secret': os.getenv('STRAVA_CLIENT_SECRET'),
        'refresh_token': os.getenv('STRAVA_REFRESH_TOKEN'),
        'grant_type': "refresh_token"
    }
    res = requests.post("https://www.strava.com/oauth/token", data=payload)
    res.raise_for_status()
    return res.json()['access_token']

def fetch_all_activities(token):
    """Récupère l'intégralité des activités avec pagination"""
    print("📡 Récupération de l'historique complet Strava...")
    url = "https://www.strava.com/api/v3/athlete/activities"
    headers = {'Authorization': f'Bearer {token}'}
    
    all_activities = []
    page = 1
    per_page = 200 # Maximum autorisé par Strava

    while True:
        
        params = {'per_page': per_page, 'page': page}
        res = requests.get(url, headers=headers, params=params)
        res.raise_for_status()
        
        data = res.json()
        
        if not data:
            break
            
        all_activities.extend(data)
        page += 1
        time.sleep(0.5)

    
    return pd.DataFrame(all_activities)

def save_to_local_data(df):
    """Sauvegarde le DataFrame en local dans le dossier data/"""
   
    
    # Ciblage du dossier data à la racine du projet
    dossier_data = ROOT_DIR / "data"
    dossier_data.mkdir(parents=True, exist_ok=True)
    
    chemin_cible = dossier_data / "activites_brutes.csv"
    
    # Sauvegarde au format CSV
    df.to_csv(chemin_cible, index=False, encoding='utf-8')
    

def extract_strava_pipeline():
    """Fonction principale d'extraction"""
    try:
        access_token = get_new_access_token()
        df_final = fetch_all_activities(access_token)
        
        if not df_final.empty:
            save_to_local_data(df_final)
        
            
    except requests.exceptions.RequestException as e:
        print(f"\n Erreur lors de l'extraction : {e}")

if __name__ == "__main__":
    extract_strava_pipeline()

📡 Récupération de l'historique complet Strava...


## Netoyage de la donnée conersion numérique intégration des indicateurs de bases pour le calcules des indices de charges : 

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Sécurisation des chemins : on remonte à la racine du projet (3 niveaux au-dessus)
# ROOT_DIR is already defined in this notebook, no need to redefine it

def nettoyer_donnees_strava(nom_fichier_entree="activites_brutes.csv", nom_fichier_sortie="activites_clean.csv"):
    """
    Lit le fichier brut, nettoie les données, crée les KPIs physiologiques
    et sauvegarde le fichier propre dans le dossier data/.
    """
    print("\n⏳ Démarrage du nettoyage des données Strava...")
    
    # 1. CIBLAGE DU DOSSIER DATA
    dossier_data = ROOT_DIR / "data"
    
    # Recherche du fichier brut (gestion du sous-dossier 'raw' vu sur ton arborescence)
    chemin_brut = dossier_data / "raw" / nom_fichier_entree
    if not chemin_brut.exists():
        chemin_brut = dossier_data / nom_fichier_entree # Fallback directement dans data/
        
    if not chemin_brut.exists():
        raise FileNotFoundError(f"❌ Le fichier brut est introuvable. Chemins testés :\n- {dossier_data / 'raw' / nom_fichier_entree}\n- {dossier_data / nom_fichier_entree}")
    
    print(f"📥 Lecture des données brutes depuis : {chemin_brut}")
    df = pd.read_csv(chemin_brut)
    
    # 2. Filtrage des courses à pied
    types_run = ['Run', 'TrailRun', 'VirtualRun']
    mots_cles = ['run', 'course', 'trail', '400', 'séance', 'entraînement']
    masque_run = (df['type'].isin(types_run)) | (df['name'].str.lower().str.contains('|'.join(mots_cles), na=False))
    masque_exclure = df['name'].str.lower().str.contains('vélo|musculation|poids|randonnée|marche', na=False)
    df_strava = df[masque_run & ~masque_exclure].copy()
    
    # 3. Conversions métriques de base
    df_strava['distance_km'] = df_strava['distance'] / 1000
    df_strava['moving_time_min'] = df_strava['moving_time'] / 60
    df_strava['average_speed_kmh'] = (df_strava['average_speed'] * 3.6).round(2)
    
    # 4. Variables temporelles
    df_temp_date = pd.to_datetime(df_strava['start_date_local'], errors='coerce')
    df_strava['jour'] = df_temp_date.dt.day_name(locale='fr_FR').str.capitalize()
    df_strava['semaine'] = df_temp_date.dt.isocalendar().week.astype(str)
    df_strava['trimestre'] = df_temp_date.dt.to_period('Q').astype(str)
    df_strava['annee'] = df_temp_date.dt.year.astype(str)
    df_strava['start_date_local'] = df_temp_date.dt.date
    df_strava = df_strava.sort_values('start_date_local')
    
    # 5. Classification des distances cibles
    conditions = [
        df_strava['distance_km'].between(4.5, 5.5),
        df_strava['distance_km'].between(9.5, 10.5),
        df_strava['distance_km'].between(20.5, 21.5),
        df_strava['distance_km'].between(41.0, 43.5)
    ]
    choices = ['5K', '10K', '21K', 'Marathon']
    df_strava['distance_cible'] = pd.Series(np.select(conditions, choices, default='Autre'), index=df_strava.index)
    
    # 6. Physiologie (VMA, Karvonen, Stress Méca)
    FC_REPOS, FC_MAX, POIDS = 45, 190, 71
    df_strava['average_heartrate'] = pd.to_numeric(df_strava.get('average_heartrate', 0), errors='coerce')
    df_strava["indice_d_effort_K"] = ((df_strava['average_heartrate'] - FC_REPOS) / (FC_MAX - FC_REPOS)).round(2)
    
    df_strava['VMA'] = np.where(df_strava["indice_d_effort_K"] > 0, 
                                (df_strava['average_speed_kmh'] / df_strava["indice_d_effort_K"]).round(2), np.nan)
    
    df_strava["Puissance_VMA"] = POIDS * df_strava["VMA"].fillna(18.0) * 0.28
    
    df_strava["average_watts"] = pd.to_numeric(df_strava.get("average_watts", 0), errors="coerce").fillna(0)
    df_strava["typologie_terrain"] = np.where(df_strava['name'].str.lower().str.contains('trail'), 'meuble', 'dure')
    df_strava["C_sol"] = np.where(df_strava["typologie_terrain"] == "dure", 1.2, 1.0)
    
    df_strava["stress_mecanique"] = (
        df_strava["moving_time_min"] * (df_strava["average_watts"] / df_strava["Puissance_VMA"].replace(0, np.nan)) ** 2 * df_strava["C_sol"]
    ).round(3)
    
    # 7. Classification Machine Learning (Intensité)
    s_cardio = np.percentile(df_strava['indice_d_effort_K'].fillna(0), [30, 80])
    s_meca = np.percentile(df_strava['stress_mecanique'].fillna(0), [30, 80])
    
    def pre_class(r):
        if r["indice_d_effort_K"] >= s_cardio[1] or r["stress_mecanique"] >= s_meca[1]: 
            return "Intense"
        elif r["indice_d_effort_K"] <= s_cardio[0] and r["stress_mecanique"] <= s_meca[0]: 
            return "Faible"
        else: 
            return "Modéré"
            
    df_strava['type_entrainement'] = df_strava.apply(pre_class, axis=1)

    # 8. NETTOYAGE FINAL ET SAUVEGARDE
    colonnes_utiles = ['name', 'start_date_local', 'annee', 'trimestre', 'distance_cible', 'distance_km', 'moving_time_min', 'VMA', 'indice_d_effort_K', 'stress_mecanique', 'type_entrainement', 'jour', 'semaine']
    df_export = df_strava[colonnes_utiles].copy()
    
    # Création explicite du fichier au même endroit
    chemin_clean = dossier_data / nom_fichier_sortie
    
    # LIGNE CRUCIALE : Force la création du dossier parent (data/) s'il n'existe pas encore
    chemin_clean.parent.mkdir(parents=True, exist_ok=True)
    
    df_export.to_csv(chemin_clean, index=False, encoding='utf-8')
    
    print(f"✅ Fichier propre généré avec succès ! \n💾 Enregistré sous : {chemin_clean}")
    
    return df_export

if __name__ == "__main__":
    nettoyer_donnees_strava()


⏳ Démarrage du nettoyage des données Strava...
📥 Lecture des données brutes depuis : C:\Users\leopa\OneDrive\Documents\ETUDE\année universitaire 2025-2027\Master CNAM\Excel\Projet_entraînement\performance_sportive\data\activites_brutes.csv
✅ Fichier propre généré avec succès ! 
💾 Enregistré sous : C:\Users\leopa\OneDrive\Documents\ETUDE\année universitaire 2025-2027\Master CNAM\Excel\Projet_entraînement\performance_sportive\data\activites_clean.csv


C:\Users\leopa\AppData\Local\Temp\ipykernel_17148\1014279139.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_strava['trimestre'] = df_temp_date.dt.to_period('Q').astype(str)


## Étape 4 : L'Export des données nétoyées. 

Une fois les données récupérées sous forme de DataFrame Pandas ous allons les sauvegarder en format CSV directement sur notre espace de stockage objet (S3/MinIO) sur Onyxia. C'est la "Zone Bronze" de notre Data Lake. 

In [ ]:
import os
import s3fs
from pathlib import Path
from dotenv import load_dotenv

# Remonte à la racine du projet

load_dotenv(dotenv_path=ROOT_DIR / ".env")

# Définition des chemins du fichier (Source locale et Destination S3)
chemin_local = ROOT_DIR / "data" / "activites_clean.csv"
chemin_s3 = "paleo/donnees_strava/activites_clean.csv"

def upload_clean_to_onyxia():
    """Envoie le fichier activites_clean.csv local vers Onyxia"""
   
    
    # Sécurité : On vérifie que le fichier a bien été créé par clean.py avant d'essayer de l'envoyer
    if not chemin_local.exists():
       
        return

    try:
        # Initialisation du client S3 (MinIO)
        fs = s3fs.S3FileSystem(
            client_kwargs={'endpoint_url': f"https://{os.getenv('AWS_S3_ENDPOINT')}"},
            key=os.getenv('AWS_ACCESS_KEY_ID'),
            secret=os.getenv('AWS_SECRET_ACCESS_KEY'),
            token=os.getenv('AWS_SESSION_TOKEN')
        )
        
        print(f"📤 Envoi en cours vers S3 : {chemin_s3}...")
        
        # 'put' prend le fichier physique de ton ordinateur et l'envoie sur le bucket
        fs.put(str(chemin_local), chemin_s3)
        
        print(f"✅ Succès ! Le fichier clean a été envoyé sur Onyxia avec succès.")
        
        except Exception as e:
        print(f"❌ Erreur lors de l'upload : {e}")

    if __name__ == "__main__":
        upload_clean_to_onyxia()
        print(f"❌ Erreur lors de l'upload : {e}")

    if __name__ == "__main__":
        upload_clean_to_onyxia()
    upload_clean_to_onyxia()

SyntaxError: invalid syntax (2922770138.py, line 39)

## Étape 5 : Test de l'Import depuis le Cloud

Pour s'assurer que notre sauvegarde a bien fonctionné, nous allons télécharger le fichier fraîchement créé depuis Onyxia et l'enregistrer dans notre dossier local `data/raw/`.

In [ ]:
def download_from_onyxia():
    """Télécharge le fichier brut depuis Onyxia vers data/raw/"""
    print("☁️ Connexion au stockage Onyxia pour le téléchargement...")
    
    # Cible le dossier data/raw à la racine
    DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
    DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

    chemin_s3 = "paleo/donnees_strava/activites_brutes.csv"
    chemin_local = DATA_RAW_DIR / "activites_brutes_test.csv"
    
    try:
        fs = s3fs.S3FileSystem(
            client_kwargs={'endpoint_url': f"https://{os.getenv('AWS_S3_ENDPOINT')}"},
            key=os.getenv('AWS_ACCESS_KEY_ID'),
            secret=os.getenv('AWS_SECRET_ACCESS_KEY'),
            token=os.getenv('AWS_SESSION_TOKEN')
        )
        
        print(f"📥 Téléchargement depuis : {chemin_s3}")
        fs.get(chemin_s3, str(chemin_local))
        print(f"✅ Succès ! Fichier test téléchargé dans : {chemin_local}")
        
    except Exception as e:
        print(f"❌ Erreur lors du téléchargement : {e}")

# Exécution du test de téléchargement
download_from_onyxia()